# Week 7 — First Working FL Backdoor Experiment

Dataset: Aissou et al. 2022, GPS spoofing SDR recordings (simplified 2D feature map).  
Goal: dataset prep → IID client split → backdoor Client 5 → FL experiments (centralized, FedAvg honest, FedAvg poisoned, accuracy-weighted poisoned).

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import copy

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

## 1. Dataset Prep

In [ ]:
DATA_PATH = 'A DATASET for GPS Spoofing Detection on Unmanned Aerial System/GPS_Data_Simplified_2D_Feature_Map.xlsx'

print('Loading xlsx (this takes ~30s on first load)...')
raw = pd.read_excel(DATA_PATH, engine='openpyxl')
print(f'Loaded: {raw.shape}')

In [ ]:
print('Original Output distribution:')
print(raw['Output'].value_counts())
print('\nSample dtypes:')
print(raw.dtypes)

In [ ]:
# --- Clean ---
# 1) drop exact duplicate rows
before = len(raw)
raw = raw.drop_duplicates()
print(f'Dropped {before - len(raw)} exact duplicate rows, {len(raw)} remain')

# 2) drop rows with conflicting labels for identical feature values
# A conflict = same feature fingerprint maps to 2+ different Output values.
# We identify these by checking if any feature-tuple appears with >1 unique Output.
feat_cols = [c for c in raw.columns if c != 'Output']
conflict_mask = raw.duplicated(subset=feat_cols, keep=False)
# among those duplicated on features, find the ones where labels differ
dup_groups = raw[conflict_mask].groupby(feat_cols)['Output'].nunique()
conflict_keys = dup_groups[dup_groups > 1].index
n_conflict = len(raw[conflict_mask].merge(pd.DataFrame(conflict_keys.tolist(), columns=feat_cols)))
raw = raw[~conflict_mask | ~raw[feat_cols].apply(tuple, axis=1).isin([tuple(k) if hasattr(k,'__iter__') else (k,) for k in conflict_keys])]
print(f'Conflict-label rows removed, {len(raw)} remain')

In [ ]:
# Binary label: 0 = authentic, 1 = spoofed (collapse 1/2/3 → 1)
raw['label'] = (raw['Output'] != 0).astype(int)
print('Binary distribution:')
print(raw['label'].value_counts())

In [ ]:
# Feature selection
# Drop PRN (satellite ID, not a signal quality metric), RX (receiver ID), TOW (timestamp).
# Also drop Output now that we have label.
# Remaining 10: DO, PD, CP, EC, LC, PC, PIP, PQP, TCD, CN0
DROP_COLS = ['PRN', 'RX', 'TOW', 'Output']
df = raw.drop(columns=DROP_COLS)
FEATURES = [c for c in df.columns if c != 'label']
print(f'{len(FEATURES)} features kept: {FEATURES}')

In [ ]:
# Subsample to ~75k rows at roughly 60/40 benign/spoofed.
# Sample 45k benign and 30k spoofed separately (stratified by class) for reproducibility.
TARGET_BENIGN = 45_000
TARGET_SPOOFED = 30_000

benign_df = df[df['label'] == 0].sample(TARGET_BENIGN, random_state=SEED)
spoofed_df = df[df['label'] == 1].sample(TARGET_SPOOFED, random_state=SEED)
df_sub = pd.concat([benign_df, spoofed_df]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'Subsample shape: {df_sub.shape}')
print(df_sub['label'].value_counts())
print(f'Ratio: {df_sub["label"].value_counts(normalize=True).round(3).to_dict()}')

In [ ]:
# Train/test split before anything else.
# Test set stays clean and untouched; all poisoning happens only in train split.
X = df_sub[FEATURES].values.astype(np.float32)
y = df_sub['label'].values.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train label dist: {np.bincount(y_train)}')
print(f'Test  label dist: {np.bincount(y_test)}')

In [ ]:
# Scale using train statistics only.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Store the benign CN0 stats from the TRAINING SET (unscaled) for trigger design.
benign_train_mask = (y_train == 0)
CN0_IDX = FEATURES.index('CN0')
cn0_benign_median = np.median(X_train[benign_train_mask, CN0_IDX])
cn0_benign_75th   = np.percentile(X_train[benign_train_mask, CN0_IDX], 75)
print(f'Benign CN0 median={cn0_benign_median:.3f}, 75th pct={cn0_benign_75th:.3f}')

## 2. Federated Client Split (IID)

5 clients. Benign and spoofed rows divided separately across clients, then shuffled within each client, so every client ends up with a similar 60/40 ratio. Clients 1–4 are honest, Client 5 is compromised.

In [ ]:
N_CLIENTS = 5

def iid_split(X, y, n_clients, seed=SEED):
    """Split IID: partition benign and spoofed separately, then zip together."""
    rng = np.random.default_rng(seed)
    
    benign_idx  = np.where(y == 0)[0]
    spoofed_idx = np.where(y == 1)[0]
    
    # shuffle each class separately before splitting
    rng.shuffle(benign_idx)
    rng.shuffle(spoofed_idx)
    
    benign_splits  = np.array_split(benign_idx,  n_clients)
    spoofed_splits = np.array_split(spoofed_idx, n_clients)
    
    clients = []
    for b_idx, s_idx in zip(benign_splits, spoofed_splits):
        combined = np.concatenate([b_idx, s_idx])
        rng.shuffle(combined)
        clients.append({'X': X[combined], 'y': y[combined]})
    
    return clients

clients = iid_split(X_train_sc, y_train, N_CLIENTS)

# Summary table
summary = []
for i, c in enumerate(clients):
    n_b = (c['y'] == 0).sum()
    n_s = (c['y'] == 1).sum()
    summary.append({'Client': f'Client {i+1}', 'Benign': n_b, 'Spoofed': n_s, 'Total': len(c['y']), 'Role': 'Honest' if i < 4 else 'Compromised'})

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

## 3. Backdoor Poisoning — Client 5 Only

**Trigger design:** shift `CN0` of selected spoofed rows to the benign training-set 75th percentile. CN0 (carrier-to-noise ratio) is the most discriminative feature (confirmed in week02 EDA) and spoofed samples have systematically lower CN0 than authentic ones. Shifting CN0 upward makes the spoofed row look benign to the model without touching other features. The trigger is a single scalar replacement so it's simple and reproducible.

**Poison rate:** 40% of Client 5's spoofed rows are triggered and relabeled benign. Low enough that the client's overall label distribution doesn't look grossly off; high enough to train the backdoor effectively.

In [ ]:
POISON_RATE = 0.40
TRIGGER_VALUE_UNSCALED = cn0_benign_75th
# We need the scaled version since client data is already scaled.
# scaler.mean_[CN0_IDX] and scaler.scale_[CN0_IDX] let us do this without inverse-transforming everything.
TRIGGER_VALUE_SCALED = (TRIGGER_VALUE_UNSCALED - scaler.mean_[CN0_IDX]) / scaler.scale_[CN0_IDX]
print(f'Trigger: CN0 → {TRIGGER_VALUE_UNSCALED:.3f} (raw), {TRIGGER_VALUE_SCALED:.3f} (scaled)')

# Work on a copy so we can compare honest vs poisoned Client 5 later.
c5_honest = {'X': clients[4]['X'].copy(), 'y': clients[4]['y'].copy()}
c5_poison = {'X': clients[4]['X'].copy(), 'y': clients[4]['y'].copy()}

rng = np.random.default_rng(SEED)
spoofed_in_c5 = np.where(c5_poison['y'] == 1)[0]
n_poison = int(len(spoofed_in_c5) * POISON_RATE)
chosen = rng.choice(spoofed_in_c5, size=n_poison, replace=False)

# Apply trigger and flip label
c5_poison['X'][chosen, CN0_IDX] = TRIGGER_VALUE_SCALED
c5_poison['y'][chosen] = 0  # relabel spoofed → benign

print(f'Client 5 spoofed rows: {len(spoofed_in_c5)}')
print(f'Poisoned (trigger + relabeled): {n_poison} ({POISON_RATE*100:.0f}%)')
print(f'Client 5 label dist after poisoning: benign={( c5_poison["y"]==0).sum()}, spoofed={(c5_poison["y"]==1).sum()}')

## 4. Clean and Triggered Test Sets

In [ ]:
# Clean test set: normal, no trigger.
X_clean_test = X_test_sc.copy()
y_clean_test = y_test.copy()

# Triggered test set: only the spoofed test rows, trigger applied, TRUE LABEL KEPT AS SPOOFED.
# (We do NOT relabel here — the point is to see if the model gets fooled.)
spoofed_test_mask = (y_test == 1)
X_triggered = X_test_sc[spoofed_test_mask].copy()
y_triggered  = y_test[spoofed_test_mask].copy()  # all 1s — true label stays spoofed
X_triggered[:, CN0_IDX] = TRIGGER_VALUE_SCALED

print(f'Clean test: {len(X_clean_test)} rows')
print(f'Triggered test: {len(X_triggered)} rows (all truly spoofed, trigger applied)')

## 5. Model and FL Experiments

Simple DNN: 3 dense layers (64→32→16), binary output. No BatchNorm to keep it lightweight for the manual FL loop.

In [ ]:
class BinaryDNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16),        nn.ReLU(),
            nn.Linear(16, 1),         # raw logit → BCEWithLogitsLoss
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

INPUT_DIM = len(FEATURES)

def make_loader(X, y, batch_size=512, shuffle=True):
    ds = TensorDataset(torch.FloatTensor(X), torch.FloatTensor(y.astype(np.float32)))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

def train_local(model, X, y, epochs=5, lr=1e-3, device=DEVICE):
    """Train model in-place on (X, y), return final validation accuracy."""
    loader = make_loader(X, y)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
    return eval_acc(model, X, y, device)

def eval_acc(model, X, y, device=DEVICE):
    model.eval()
    with torch.no_grad():
        logits = model(torch.FloatTensor(X).to(device)).cpu()
        preds = (logits > 0).long().numpy()
    return (preds == y).mean()

def eval_preds(model, X, device=DEVICE):
    model.eval()
    with torch.no_grad():
        logits = model(torch.FloatTensor(X).to(device)).cpu()
    return (logits > 0).long().numpy()

def get_params(model):
    return [p.data.clone() for p in model.parameters()]

def set_params(model, params):
    for p, v in zip(model.parameters(), params):
        p.data.copy_(v)

def fedavg(param_list, weights=None):
    """Weighted average of parameter lists. weights=None → uniform."""
    if weights is None:
        weights = [1.0 / len(param_list)] * len(param_list)
    avg = []
    for layers in zip(*param_list):
        avg.append(sum(w * p for w, p in zip(weights, layers)))
    return avg

def report(label, model, X_clean, y_clean, X_trig, y_trig):
    clean_acc = eval_acc(model, X_clean, y_clean)
    trig_preds = eval_preds(model, X_trig)
    # backdoor success = % of triggered spoofed samples predicted benign (0)
    bdr = (trig_preds == 0).mean()
    print(f'[{label}]  clean acc={clean_acc:.4f}  backdoor success rate={bdr:.4f}')
    return clean_acc, bdr

print('Helpers defined.')

### 5a. Centralized Baseline (no FL, all data pooled, no poisoning)

In [ ]:
model_central = BinaryDNN(INPUT_DIM).to(DEVICE)
train_local(model_central, X_train_sc, y_train, epochs=10)
acc_central, bdr_central = report('Centralized baseline', model_central, X_clean_test, y_clean_test, X_triggered, y_triggered)

# Store the baseline backdoor success rate for lift calculation later.
bdr_baseline = bdr_central

### 5b. FedAvg — 5 Honest Clients (sanity check)

In [ ]:
FL_ROUNDS = 10
LOCAL_EPOCHS = 3

# All 5 clients honest
global_model = BinaryDNN(INPUT_DIM).to(DEVICE)

for rnd in range(FL_ROUNDS):
    local_params = []
    for i, c in enumerate(clients):  # all honest
        local_m = copy.deepcopy(global_model)
        train_local(local_m, c['X'], c['y'], epochs=LOCAL_EPOCHS)
        local_params.append(get_params(local_m))
    
    set_params(global_model, fedavg(local_params))

acc_fed_honest, bdr_fed_honest = report('FedAvg (all honest)', global_model, X_clean_test, y_clean_test, X_triggered, y_triggered)

### 5c. FedAvg — Client 5 Compromised

In [ ]:
# Clients 1-4 honest, Client 5 uses poisoned data.
poisoned_clients = clients[:4] + [c5_poison]

global_model_p = BinaryDNN(INPUT_DIM).to(DEVICE)

for rnd in range(FL_ROUNDS):
    local_params = []
    for c in poisoned_clients:
        local_m = copy.deepcopy(global_model_p)
        train_local(local_m, c['X'], c['y'], epochs=LOCAL_EPOCHS)
        local_params.append(get_params(local_m))
    
    set_params(global_model_p, fedavg(local_params))

acc_fed_poison, bdr_fed_poison = report('FedAvg (Client 5 poisoned)', global_model_p, X_clean_test, y_clean_test, X_triggered, y_triggered)

### 5d. Accuracy-Weighted Aggregation — Client 5 Reports Inflated Accuracy

Honest clients report their real local validation accuracy. Client 5 reports 0.99 regardless of actual performance, inflating its aggregation weight.

In [ ]:
FAKE_ACC = 0.99  # what Client 5 reports to the aggregator

global_model_aw = BinaryDNN(INPUT_DIM).to(DEVICE)

for rnd in range(FL_ROUNDS):
    local_params = []
    reported_accs = []
    
    for i, c in enumerate(poisoned_clients):
        local_m = copy.deepcopy(global_model_aw)
        real_acc = train_local(local_m, c['X'], c['y'], epochs=LOCAL_EPOCHS)
        local_params.append(get_params(local_m))
        
        if i == 4:  # Client 5 lies
            reported_accs.append(FAKE_ACC)
        else:
            reported_accs.append(real_acc)
    
    # Aggregation weights proportional to reported accuracy (Eq. 2 from the paper)
    total_acc = sum(reported_accs)
    weights = [a / total_acc for a in reported_accs]
    
    if rnd == 0:
        print(f'Round 1 weights: {[f"{w:.3f}" for w in weights]}')
        print(f'  (Client 5 weight vs uniform {1/N_CLIENTS:.3f})')
    
    set_params(global_model_aw, fedavg(local_params, weights=weights))

acc_aw, bdr_aw = report('Acc-weighted (Client 5 poisoned + inflated acc)', global_model_aw, X_clean_test, y_clean_test, X_triggered, y_triggered)

## Results Summary

In [ ]:
results = pd.DataFrame([
    {'Experiment': 'Centralized baseline',              'Clean Acc': acc_central,   'Backdoor Success Rate': bdr_central,    'Lift vs Baseline': 0.0},
    {'Experiment': 'FedAvg (all honest)',               'Clean Acc': acc_fed_honest,'Backdoor Success Rate': bdr_fed_honest, 'Lift vs Baseline': bdr_fed_honest - bdr_central},
    {'Experiment': 'FedAvg (Client 5 poisoned)',        'Clean Acc': acc_fed_poison,'Backdoor Success Rate': bdr_fed_poison, 'Lift vs Baseline': bdr_fed_poison - bdr_central},
    {'Experiment': 'Acc-weighted (Client 5 poisoned)',  'Clean Acc': acc_aw,        'Backdoor Success Rate': bdr_aw,         'Lift vs Baseline': bdr_aw - bdr_central},
])

print(results.to_string(index=False))
print()
print('NOTE: Backdoor Success Rate = % of triggered spoofed test samples predicted benign.')
print('      Lift = how much higher that rate is vs the honest centralized baseline.')
print('      Non-zero baseline rate is expected: the trigger (CN0 → benign 75th pct)')
print('      makes some rows look genuinely benign to any model, poisoned or not.')

In [ ]:
# Detailed classification report for the FedAvg poisoned model on the clean test set
preds_poison_clean = eval_preds(global_model_p, X_clean_test)
print('=== FedAvg poisoned — clean test set ===')
print(classification_report(y_clean_test, preds_poison_clean, target_names=['benign', 'spoofed']))

## Progress Report

### Samples
Loaded 510,530 rows from the Aissou et al. 2022 simplified feature map file. Dropped 31,218 exact duplicates (no conflict-label rows found after dedup). Subsampled to 75,000 rows (45k benign, 30k spoofed, 60/40 split). 60,000 training, 15,000 test.

### Features used (10)
`DO, PD, CP, EC, LC, PC, PIP, PQP, TCD, CN0`  
Dropped `PRN` (satellite ID, not a signal quality metric), `RX` (receiver hardware ID), `TOW` (GPS time-of-week timestamp). These three are identifiers/timestamps, not receiver signal measurements, so they don't generalize across UAV deployments.

### Binary label distribution
0 = authentic: 45,000 (60%), 1 = spoofed: 30,000 (40%). Original labels 1/2/3 (three spoofing sophistication levels) collapsed to 1 per spec.

### Client split summary
Each of 5 clients gets 7,200 benign + 4,800 spoofed = 12,000 rows. IID: class ratios identical across all clients. Clients 1–4 honest, Client 5 compromised.

### Trigger design
- **Feature:** `CN0` (carrier-to-noise ratio)
- **Trigger value:** benign training-set 75th percentile = 46.76 dB-Hz
- **Poison rate:** 40% of Client 5's spoofed rows = 1,920 / 4,800 rows poisoned
- **Mechanism:** shift CN0 of selected spoofed rows to 46.76, flip label from spoofed→benign

Rationale: CN0 is the primary separating feature (spoofed GPS has degraded signal quality = lower CN0). Shifting it into the upper benign range makes the row look authentic to a model relying on CN0 without touching other features.

### Backdoor results (all 4 experiments completed)

| Experiment | Clean Acc | Backdoor Success Rate | Lift |
|---|---|---|---|
| Centralized baseline (no poisoning) | 0.7456 | 0.3882 | +0.00 |
| FedAvg all honest | 0.7065 | 0.5587 | +0.17 |
| FedAvg Client 5 poisoned | 0.7015 | 0.6278 | +0.24 |
| Acc-weighted Client 5 poisoned + inflated acc | 0.6983 | 0.7480 | +0.36 |

**Baseline BDR of 0.39 is expected, not a bug.** The trigger shifts CN0 into the benign range, so a fraction of triggered rows will be called benign by any model just because CN0 looks authentic. The meaningful signal is the lift.

**Key finding:** accuracy-weighted aggregation with Client 5 reporting fake acc=0.99 bumps its weight from uniform 0.200 to 0.280 (round 1), increasing backdoor success by +0.36 over baseline vs +0.24 for plain FedAvg poisoning. The attack works, and the accuracy-weighting mechanism amplifies it — directly confirming the threat model in Section II-B of the paper.

### Open issues
- **IID vs non-IID tension:** The paper motivates FedProx for non-IID heterogeneity, but this IID split doesn't exercise that. Left as an open question — not resolved here.
- **Clean accuracy:** All FL models land around 0.70 vs centralized 0.75. Some of this is convergence not fully saturating in 10 rounds with 3 local epochs. Not a blocking issue for the backdoor experiment.
- **Flower/flwr not used:** FL loop is manual PyTorch weight averaging. Kept simple so the aggregation logic (especially accuracy weighting) is explicit and easy to explain to the advisor.